In [2]:
import matplotlib
matplotlib.use("Agg")
import numpy as np
import matplotlib.pyplot as plt
import h5py

import importlib
import os
import sys
import re
import glob

In [3]:
path_openket = "//home//ultrxvioletx//GitHub//openket"
if path_openket not in sys.path:
    sys.path.append(path_openket)
# elimina el caché de func.py para que no lea los datos anteriores
if 'func' in sys.modules:
    del sys.modules['func']
if os.path.exists('func.py'):
    os.remove('func.py')

from openket.core.diracobject import Ket, Bra, Operator, AnnihilationOperator, CreationOperator
from openket.core.evolution import build_ode, sym2num, init_state
from openket.core.metrics import dag, comm, ptrace, trace, normalize, sub_qexpr, op2dict, qmatrix

import style
importlib.reload(style)
from style import *
set_style()

2026-03-30 17:34:47,894 - openket - INFO - openket v0.1.0 initialized successfully.


##### funciones auxiliares

In [4]:
def convergencia(tiempo, fotones, tol=1e-3, porcentaje=0.1):
    npoints = int(len(tiempo) * porcentaje)
    if npoints < 2:
        return False, np.nan # no hay suficientes puntos
    fotones = fotones[-npoints:]
    tiempo = tiempo[-npoints:]
    df = np.gradient(fotones, tiempo)
    df_mean = np.mean(np.abs(df))
    return df_mean < tol, df_mean

In [5]:
def procesafile(at, lvl, folder, barrido="detuning_pa"):
    global detunings, Nss, non_converged, Probs, niveles, attrs
    detunings = []
    Probs = [[] for _ in np.arange(lvl**at)]
    Nss = []
    attrs = {}
    
    non_converged = []
    files = glob.glob(os.path.join(path,'*.h5'))
    for i,file in enumerate(files):
        with h5py.File(file, 'r') as f:
            dataset = os.path.basename(file).replace('.h5', '')
            if i == 0:
                attrs = dict(f[dataset].attrs)
            
            detuning = f[dataset].attrs[barrido]
            tt = f[dataset].attrs['t']
            niveles = f[dataset].attrs['probs_label']
            tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))
    
            if dataset not in f:
                print(f"  no se encontró el dataset '{dataset}' en el archivo. Saltando.")
                continue

            detunings.append(detuning)
            rho = f[dataset][:]
            lvl_prob = [rho[:, i] for i in np.arange(lvl**at)] # las primeras n columnas son las probabilidades
            N_expect = rho[:, lvl**at] # la última columna es el valor medio de fotones en la cavidad
            N_expect = np.real(N_expect)

            # promediamos el último 25% de los puntos
            npoints = int(rho.shape[0] * 0.25)
            Nss.append(np.mean(N_expect[-npoints:]))
            for i,ele in enumerate(lvl_prob):
                Probs[i].append(np.mean(ele[-npoints:]))
            
            is_converged, derivative = convergencia(tiempo, N_expect)
            if not is_converged:
                non_converged.append([detuning, np.mean(N_expect[-npoints:])])
                
    # ordena por detuning para mejorar la gráfica
    detunings = np.array(detunings)
    sorti = np.argsort(detunings)
    
    detunings = detunings[sorti]
    Nss = np.array(Nss)[sorti]
    Probs = [np.real(np.array(p))[sorti] for p in Probs]

## 1 átomo 4 niveles

#### CON rabi

In [6]:
at,lvl = 1,4
path = os.path.join("detunings",f"{at}at{lvl}lvl","conrabi")

procesafile(at, lvl, path)

plt.plot(detunings, Nss, "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
#plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.01, ymax=0.03)
plt.savefig(f"../figuras/1at3lvl_fotones.png")
plt.close()

In [8]:
plvls = ["e", "s"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    
    plt.plot(detunings, prob, "s-", label=rf"${latex["P"]}{plvl}$", markersize=5, linewidth=1, color=colores["estados"][j])
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.01, ymax=0.03)
plt.legend()
plt.savefig(f"../figuras/1at3lvl_excitado.png")
plt.close()

In [7]:
attrs

{'detuning_ac': np.float64(0.0),
 'detuning_pa': np.float64(-3.2),
 'g': np.float64(3.0),
 'gamma_e': np.float64(1.0),
 'gamma_s': np.float64(0.01),
 'id': np.int32(19),
 'kappa': np.float64(1.0),
 'n': np.int32(5),
 'probs_label': array(['g', 'e', 's', 'p'], dtype=object),
 'rabi_c': np.float64(2.0),
 'rabi_p': np.float64(0.1),
 't': array(['0', '15', '1000'], dtype=object)}

#### SIN rabi

In [9]:
at,lvl = 1,4
path = os.path.join("detunings",f"{at}at{lvl}lvl","sinrabi")

procesafile(at, lvl, path)

plt.plot(detunings, Nss, "s-", markersize=5, linewidth=1, color=colores["fotones"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
#plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.005, ymax=0.01)
plt.savefig(f"../figuras/1at3lvl_fotones0.png")
plt.close()

In [11]:
plvls = ["e", "s"]
for j,plvl in enumerate(plvls):
    pattern_str = plvl.replace("_", ".")
    pattern_str = pattern_str[:at] # recorta al largo del dataset
    pattern = re.compile(f"^{pattern_str}$")
    indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    prob = np.sum([Probs[i] for i in indices], axis=0)
    
    plt.plot(detunings, prob, "s-", label=rf"${latex["P"]}{plvl}$", markersize=5, linewidth=1, color=colores["estados"][j])
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel(rf"${latex["Dpa"]}$")
plt.ylabel(r"Probabilidad de excitación")
ax = plt.gca()
format_ax(ax, xstep=1, ystep=0.005, ymax=0.01)
plt.legend()
plt.savefig(f"../figuras/1at3lvl_excitado0.png")
plt.close()

In [12]:
attrs

{'detuning_ac': np.float64(0.0),
 'detuning_pa': np.float64(-3.2),
 'g': np.float64(3.0),
 'gamma_e': np.float64(1.0),
 'gamma_s': np.float64(0.01),
 'id': np.int32(19),
 'kappa': np.float64(1.0),
 'n': np.int32(5),
 'probs_label': array(['g', 'e', 's', 'p'], dtype=object),
 'rabi_c': np.float64(0.0),
 'rabi_p': np.float64(0.1),
 't': array(['0', '15', '1000'], dtype=object)}

## 2 átomos 4 niveles

### omegas

In [33]:
at,lvl = 2,4
path = os.path.join("omegas",f"{at}at{lvl}lvl")

procesafile(at, lvl, path, barrido="Omega_EE")

cut = 2
mask = (detunings >= -cut) & (detunings <= cut)
plt.plot(detunings[mask], Nss[mask], "s-", markersize=5, linewidth=1, color=colores["fotones_ss"])
if non_converged:
    dtn = [ele[0] for ele in non_converged]
    nmean = [ele[1] for ele in non_converged]
    plt.scatter(dtn, nmean, color="red")
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel(rf"${latex["Oee"]}$")
plt.ylabel(rf"Número medio de fotones ${latex["Nss"]}$")
#ax = plt.gca()
#format_ax(ax, xstep=1, ystep=0.01, ymax=0.03)
plt.savefig(f"../figuras/fig:omegas_2at4lvl_fotones.png")
plt.close()

plvls = ["ss", "s"]
labels = ["{ss}", "s"]
for j,plvl in enumerate(plvls):
    if '_' in plvl:
        # importa la posición: "_s" -> "^.s$", "s_" -> "^s.$"
        pattern_str = plvl.replace("_", ".")
        pattern_str = pattern_str[:at] # recorta al largo del dataset
        pattern = re.compile(f"^{pattern_str}$")
        indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    else:
        # sin posición: "s" -> niveles que contengan exactamente una "s"
        indices = [i for i, lbl in enumerate(niveles) if plvl in lbl and lbl.count(plvl) == 1]
    
    prob = np.sum([Probs[i] for i in indices], axis=0)
    
    plt.plot(detunings[mask], prob[mask], "s-", label=rf"${latex["P"]}{labels[j]}$", markersize=5, linewidth=1, color=colores["estados_ss"][j])
plt.xlabel(rf"${latex["Oee"]}$")
plt.ylabel(r"Probabilidad de excitación")
#ax = plt.gca()
#format_ax(ax, xstep=2)
plt.legend()
plt.savefig(f"../figuras/fig:omegas_2at4lvl_excitado.png")
plt.close()

In [14]:
attrs

{'Omega_DR': np.float64(0.0),
 'Omega_EE': np.float64(2.0),
 'detuning_al': np.float64(0.0),
 'detuning_bc': np.float64(0.0),
 'detuning_ca': np.float64(0.0),
 'g': np.float64(3.0),
 'gamma_12': np.float64(0.0),
 'gamma_e': np.float64(1.0),
 'gamma_s': np.float64(0.01),
 'id': np.int32(5),
 'kappa': np.float64(1.0),
 'n': np.int32(5),
 'probs_label': array(['gg', 'ge', 'gs', 'gp', 'eg', 'ee', 'es', 'ep', 'sg', 'se', 'ss',
        'sp', 'pg', 'pe', 'ps', 'pp'], dtype=object),
 'rabi_c': np.float64(2.0),
 'rabi_p': np.float64(0.1),
 't': array(['0', '50', '1000'], dtype=object)}

### oscilaciones sin cavidad

In [22]:
from scipy.signal import find_peaks
def f_dominante(signal, tiempo):
    freq = None
    peaks, _ = find_peaks(signal, height=np.max(signal)*0.5)  # altura mínima 50% del máximo
    if len(peaks) > 1:
        T = tiempo[peaks[1]] - tiempo[peaks[0]]
        freq = 1 / T
    return freq

In [16]:
def procesafile2(at, lvl, path):
    barrido = []
    Probs = [[] for _ in np.arange(lvl**at)]
    label = f"{at}at{lvl}lvl"
    title = ""
    niveles = None
    tiempo = None
    has_omega = (at > 1) # 1 átomo no tiene Omega_EE

    files = glob.glob(os.path.join(path, label + '*.h5'))
    for file in files:
        with h5py.File(file, 'r') as f:
            dataset = os.path.basename(file).replace('.h5', '')

            d = f[dataset]
            attrs = {k: d.attrs.get(k) for k in d.attrs}
            
            niveles = f[dataset].attrs['probs_label']
            tt = f[dataset].attrs['t']
            tiempo = np.linspace(float(tt[0]), float(tt[1]), int(tt[2]))

            Omega_EE = f[dataset].attrs['Omega_EE'] if has_omega else 0.0
            barrido.append(Omega_EE)

            rho = f[dataset][:]
            lvl_prob = [rho[:, i] for i in np.arange(lvl**at)]
            for i, ele in enumerate(lvl_prob):
                Probs[i].append(ele)

    # ordena por Omega_EE
    barrido = np.array(barrido)
    sorti   = np.argsort(barrido)
    barrido = barrido[sorti]
    Probs   = [np.array(p)[sorti] for p in Probs]

    return {"barrido": barrido, "Probs": Probs, "niveles": niveles, "tiempo": tiempo, "title": title, "at": at, "has_omega": has_omega}

In [28]:
lvl, plvl = 4, "s"
omega_filter = 20.0
path = os.path.join("asaf",f"{lvl}lvl_distancias")

ds_1at = procesafile2(1, lvl, path)
ds_2at = procesafile2(2, lvl, path)
datasets = [ds_1at, ds_2at]
freqs = {}
step=10

for k,ds in enumerate(datasets):
    barrido = ds["barrido"]
    Probs = ds["Probs"]
    niveles = ds["niveles"]
    tiempo = ds["tiempo"]
    at = ds["at"]

    if '_' in plvl:
        # importa la posición: "_s" -> "^.s$", "s_" -> "^s.$"
        pattern_str = plvl.replace("_", ".")
        pattern_str = pattern_str[:at] # recorta al largo del dataset
        pattern = re.compile(f"^{pattern_str}$")
        indices = [i for i, lbl in enumerate(niveles) if pattern.search(lbl)]
    else:
        # sin posición: "s" -> niveles que contengan exactamente una "s"
        indices = [i for i, lbl in enumerate(niveles) if plvl in lbl and lbl.count(plvl) == 1]
    prob = np.sum([Probs[i] for i in indices], axis=0)  # shape: (n_omega, n_tiempo)
    if ds["has_omega"]:
        for j, dist in enumerate(barrido):
            # si hay filtro de omega, saltar los que no coincidan
            if omega_filter is not None and not np.isclose(dist, omega_filter):
                continue
            plt.plot(tiempo[::step], prob[j][::step], "s-", label=rf"{at}at; ${latex["Oee"]}={dist}$", markersize=5, linewidth=1, color=colores["estados_t"][k])
            freqs["2at"] = f_dominante(prob[j], tiempo)
    else:
        plt.plot(tiempo[::step], prob[0][::step], "s-", label=rf"{at}at", markersize=5, linewidth=1, color=colores["estados_t"][k])
        freqs["1at"] = f_dominante(prob[0], tiempo)

plt.xlabel(rf"${latex["rabip"]}$")
plt.ylabel(rf'Probabilidad de excitación')
#ax = plt.gca()
#format_ax(ax, xstep=2, ystep=2)
plt.legend()
plt.savefig(f"../figuras/fig:rabi_2at4lvl_excitado.png")
plt.close()

In [25]:
freqs["2at"] / freqs["1at"]

np.float64(1.268571428571429)